# 2  - E2E Machine Learning proiektua

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1 - Datuak lortu

* **California Housing Prices**: *Median house prices for California districts derived from the 1990 census.*

In [ ]:
# Datu basearen esteka
url = "https://github.com/ageron/data/raw/main/housing.tgz"

# jaitsi eta deskonprimatu
!curl -L {url} | tar -xz

* `!`: zerbitzariko ingurunean (*Linux shell*) exekutatuko den komandoa
* `curl` : *Transfer data from or to a server using URLs*
   * `-L` : *Follow re-location*  
* `|` (*pipe*): komando baten irteera hurrengo komandoaren sarrerarekin konektatu
* `tar` : *An archiving utility* (edukiak fitxategi bakarrean bildu)
   * `-x` : *Extract files from an archive*
   *  `-z` : *Filter the archive through gzip*

## 2 - Dataframe-a sortu eta bistaratu

CSV (Comma-Separated Values) formatuan dagoen taula **pandas**-eko *dataframe* bilakatu:

In [ ]:
df = pd.read_csv("housing/housing.csv")

`df` dataframea bistaratu (pandas-ek zenbaki osoz osotutako indizea gehitzen dio):

In [ ]:
df

Dataframeak indexatzeko aukera ezberdinak ditugu. Indexazioaren emaitza ilara/zutabe bakarra bada, emaitza `Series` motako objektu bat izango da. Beste kasuetan emaitza `DataFrame` motako objektu bat izango da:

In [ ]:
data = df["total_bedrooms"]
print(type(data))
data

In [ ]:
data = df[["total_bedrooms","population"]]
print(type(data))
data

In [ ]:
data = df.iloc[0]
print(type(data))
data

In [ ]:
data = df.iloc[10:20] # indizeko balioak mantendu egiten dira
print(type(data))
data

## 3 - Datuak aztertu

* **longitude/latitude** : barrutiaren kokapena
* **housing_median_age**: barrutiko etxeen antzinatasunaren mediana
* **total_rooms**: gela kopuru totala (barrutiko etxe guztientzat)
* **total_bedrooms**: logela kopuru totala (barrutiko etxe guztientzat)
* **population**: barrutiko biztanle kopurua
* **households**: barrutiko etxe kopurua
* **median_income**: barrutiko biztanleen diru-sarreren mediana
* **median_house_value**: barrutiko etxeen prezioen mediana
* **ocean_proximity**: ozeanoarekiko hurbiltasuna

[`dataframe.info()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) metodoak zutabeen laburpen sinple bat eskaintzen du:

In [ ]:
df.info()

[`dataframe.hist()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html) metodoak zutabeen histogramak marrazten ditu:

In [ ]:
df.hist();

`df.hist();` vs `df.hist()` : Jupyter Notebook-ek gelaxkako azken adierazpenaren itzulera-balioa testu gisa erakusten du defektuz. `;` jartzea adierazpen horren irteera automatikoa ezkutatzen du.

`hist(..)` metodoaren argumentu batzuk:
* `bins`: Number of histogram bins to be used
* `figsize`: The size in inches of the figure to create

In [ ]:
df.hist(bins=50, figsize=(15,10));

Histogramal behatzean:
* **housing_median_age**
    * $[1,52]$ tartera mugatua (52-n datu *arraroa*)
* **median_income**:
    * Eskala: ez dago dolarretan (\$10k)
    * $[0.5,15]$ tartera mugatua (\$150000etan datu *arraroa*)
* **median_house_value**
    * Eskala: dolarretan
    * $[15k,500k]$ tartera mugatua (\$500k-n datu *arraroa*) &rarr; helburu aldagaia bada, arazoa?
* Ezaugarriek eskala nahiko ezberdinak
* Orohar, banaketak ez dira simetrikoak (eskuineko alborapena dute) 

In [ ]:
for c in ['housing_median_age', 'median_income', 'median_house_value']:
    print(f'{c} ∈ [{df[c].min()} , {df[c].max()}]')

Longitude/latitude ezaugarriak barrutien kokaguneak marrazteko erabil ditzakegu:

In [ ]:
df.plot(x='longitude', y='latitude', kind='scatter')

Informazio gehiago gehitu dezakegu:
* Barruti bakoitzaren **populazioa**: puntuaren **tamaina**
* Batazbesteko etxeen **prezioa**: puntuaren **kolorea** 

In [ ]:
df.plot(
    kind='scatter',
    x='longitude', y='latitude',
    s=df['population']/200,
    c='median_house_value', cmap='jet', colorbar=True
)

Eta Californiako mapa batean kokatuko bagenu informazio guztia?

In [ ]:
# irudiaren esteka
url = "https://github.com/mpenagar/Egungo-Programazio-Teknikak/blob/master/img/california.png?raw=true"
# jaitsi 
!curl -L {url} > california.png

In [ ]:
df.plot(
    kind='scatter',
    x='longitude', y='latitude',
    s=df['population']/200,
    c='median_house_value', cmap='jet', colorbar=True
)
california_img = plt.imread('california.png')
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)

plt.show()

## 4 - Train/Test banaketa egin

Testerako datuen %20a erreserbatuko dugu. *Scikit-learn*-eko `train_test_split()` funtzioa erabiliko dugu:

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f'{df.shape = }')
print(f'{train.shape = }')
print(f'{test.shape = }')
#train
#test

* `random_state=...`: exekutatzen dugun bakoitzean sasi-auzazkotasun generatzaileak hazi berdina erabiliko du, **erreproduzigarritasuna** bermatuz.
* `42`: edozein balio izan daiteke ([El sentido de la vida, el universo y todo lo demás: 42](https://es.wikipedia.org/wiki/El_sentido_de_la_vida,_el_universo_y_todo_lo_dem%C3%A1s)) https://www.youtube.com/watch?v=W5dPnXaMcWw

Train/Test banaketa egitean, jatorrizko indizea mantentzen da:

In [ ]:
train.head()

`.reset_index(drop=True)` funtzioak indizea berritzen du.
* `drop=True` : defektuz indize zaharra ezaugarri gisa mantentzen du.

In [ ]:
train.reset_index(drop=True).head()

In [ ]:
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
#train
#test

## 5 - Aldagai iragarleak eta helburu-aldagaia ezberdindu 

Oraingoz:
* Helburu-aldagaia: **median_house_value**
* Aldagai iragarleak: beste guztiak

In [ ]:
X_train = train.drop(columns=['median_house_value'])
y_train = train['median_house_value'].copy() # bertsio berrietan .copy() ez da beharrezkoa, Copy-on-Write
#X_train
#y_train

**Copy-on-Write**: `train['median_house_value']`-k zutabe originalaren *view* bat bueltatzen du: `y_train`-en aldaketak egitean, `train`-eko zutabean aldaketak egingo genituzke. Pandas-en bertsio berriek (≥3.0) *Copy-on-Write* propietatea dute, `y_train` aldatzera goazenean datuen kopia bat sortzen da eta `train`-eko zutabean ez da aldaketarik gertatzen. 

Entrenamendu-multzoan aplikatzen diren eraldaketa guztiak testean ere aplikatu behar dira, beraz errazena da DataFrame bati aplikatu beharreko eraldaketa guztiak funtzio baten bidez adieraztea:

In [ ]:
def preprocessing(df):
    X = df.drop(columns=['median_house_value'])
    # y = df['median_house_value'].copy()
    # DataFrameak helburu-aldagairik ez balu...
    y = df['median_house_value'].copy() if 'median_house_value' in df.columns else None
    return X,y

X_train, y_train = preprocessing(train)
X_test, y_test = preprocessing(test)
print(f'{train.shape = }  {X_train.shape = }  {y_train.shape = }')
print(f'{test.shape = }  {X_test.shape = }  {y_test.shape = }')
#X_train
#y_train
#X_test
#y_test

## 7 - Ezaugarrien konbinaketa

Ezaugarrietako batzuk barruti bakoitzeko balio totalak dira (barrutian dauden etxe kopuruarekin handitzen dira) eta ez dute erabilgarritasun handirik barruti batetako ezaugarri gisa:

* **total_rooms**: gela kopuru totala (barrutiko etxe guztientzat)
* **total_bedrooms**: logela kopuru totala (barrutiko etxe guztientzat)

Zentzuzkoagoa litzateke etxebizitza baten batazbesteko gela eta logela kopurua izatea ezaugarria:
* **rooms_ph** (*average rooms per household*): `total_rooms/households`
* **bedrooms_ph** (*average rooms per household*): `total_bedrooms/households`

Ezaugarri originalak ezabatu ditzakegu:

In [ ]:
def preprocessing(df):
    df = df.copy() # ematen diguten dataframe-a ez aldatzeko...
    
    df['rooms_ph'] = df['total_rooms'] / df['households']
    df['bedrooms_ph'] = df['total_bedrooms'] / df['households']
    df = df.drop(columns=['total_bedrooms','total_rooms'])
    
    X = df.drop(columns=['median_house_value'])
    y = df['median_house_value'].copy() if 'median_house_value' in df.columns else None
    return X,y

X_train, y_train = preprocessing(train)
X_test, y_test = preprocessing(test)
print(f'{train.shape = }  {X_train.shape = }  {y_train.shape = }')
print(f'{test.shape = }  {X_test.shape = }  {y_test.shape = }')
#X_train
#y_train
#X_test
#y_test

## 6 - Balio galduak 

Balio galduak edonon agertu daitezke:

In [ ]:
X_train.isnull().sum()

Test-eko datuetan zer gertatuko den ezin dugu aurrez jakin, hala ere hemen tranpa txiki bat egingo dugu:

In [ ]:
X_test.isnull().sum()

Balio galdu hauen iturria *total_bedrooms* ezaugarria izan da:

In [ ]:
df.isnull().sum()

Aldagai kuantitatiboetan ohikoa da balio galduen ordezkapenean mediana erabiltzea.
* Mediana **train-eko datuetan zenbatetsi behar da**
* Bi fase:
   * Medianaren estimazioa (**train**) &rarr; aurreprozesatzearen barne parametro bat
   * Aurreprozesatzea aplikatzea (**train+test**)

**OHARRA:** Hau ezaugarri guztiekin egin beharko genuke, izan ere ezin dugu ziurtatu zer gerta daitekeen aurrerago. Hala ere, kasu honetan `bedrooms_ph` aldagaiean aztertuko dugu balio galduen problema.

*Scikit-learn*-ek eraldaketak doitzen/zenbatesten eta aplikatzen dituzten **transformer**-ak ditu. Orohar onako moduan erabiltzen dira:

```python
transformer = SomeTransformer()
transformer.fit(train_data)
train_data_new = transformer.transform(train_data)
test_data_new = transformer.transform(test_data)
```

Gure datuen aurreprozesatze guztiaz arduratzen den antzeko objektu bat sor genezake:

```python
class Preprocessor(object):
    def __init__(self):
        # fit() egitean doitu beharreko barne parametroak sortu
        ...
        
    def fit(self, df):
        # barne parametroak doitu
        ...
        return self # konbentzioz, fit-ek self bueltatzen du
        
    def transform(self, df):
        df = df.copy() # ematen diguten dataframe-a ez aldatzeko...
        # Egin beharreko transformazio guztiak aplikatu
        ...
        # X eta y bereiztu
        ...
        return X, y
```

Eta erabiltzeko:

```python
preprocessor = Preprocessor()
preprocessor.fit(train)
X_train, y_train = preprocessor.transform(train)
X_test, y_test = preprocessor.transform(test)
```

### Preprocessor klasearen inplementazioaren arazoa

`rooms_ph`-ren mediana zenbatestera goazenean, datuak ez daude prest (`fit` `transform` aurretik exekutatzen da):
* `rooms_ph` eta `bedrooms_ph` **ez dira existitzen**

```python
class Preprocessor:
    def __init__(self):
        # fit() egitean doitu beharreko barne parametroak sortu
        self.bedrooms_ph_median = None
    
    def fit(self, df):
        # barne parametroak doitu
        self.bedrooms_ph_median = df['bedrooms_ph'].median()
        return self # konbentzioz, fit-ek self bueltatzen du
        
    def transform(self, df):
        df = df.copy() # ematen diguten dataframe-a ez aldatzeko...
        # Egin beharreko transformazio guztiak aplikatu
        df['rooms_ph'] = df['total_rooms'] / df['households']
        df['bedrooms_ph'] = df['total_bedrooms'] / df['households']
        df = df.drop(columns=['total_bedrooms','total_rooms'])
        df['bedrooms_ph'] = df['bedrooms_ph'].fillna(self.bedrooms_ph_median)
        # X eta y bereiztu
        X = df.drop(columns=['median_house_value'])
        y = df['median_house_value'].copy() if 'median_house_value' in df.columns else None
        return X, y
```

Bigarren transformazio fasearen (*Balio galduak*) barne parametroak doitzeko (`fit`) aurreko fasearen (*Ezaugarrien konbinaketa*) transformazioa dagoeneko aplikatua egon behar da. Orohar, $i$. fasearen barne parametroak doitzeko $1,2,\dots,i-1$ faseen transformazioa dagoeneko aplikatuta egotea behar da.

Problema hau ebazteko, transformazio fase bakoitza bere *transformer* clase propioarekin adierazi dezakegu eta aurreprozesatze guztia egingo duen `Preprocessor` *transformer*-a haien menpe jarri:

### 1. `FeatureCombinator` : Ezaugarrien konbinaketa

In [ ]:
class FeatureCombinator:
    def __init__(self):
        pass
        
    def fit(self, df):
        return self
        
    def transform(self, df):
        df['rooms_ph'] = df['total_rooms'] / df['households']
        df['bedrooms_ph'] = df['total_bedrooms'] / df['households']
        df = df.drop(columns=['total_bedrooms','total_rooms'])
        return df

### 2. `MedianFiller` : Balio galduak

In [ ]:
class MedianFiller:
    def __init__(self):
        self.bedrooms_ph_median = None
        
    def fit(self, df):
        self.bedrooms_ph_median = df['bedrooms_ph'].median()
        return self
        
    def transform(self, df):
        df['bedrooms_ph'] = df['bedrooms_ph'].fillna(self.bedrooms_ph_median)
        return df

### 3. `Preprocessor` : Aurreprozesatze osoa

In [ ]:
class Preprocessor:
    def __init__(self, transformers):
        self.transformers = transformers
            
    def fit(self, df):
        df = df.copy()
        for t in self.transformers:
            # t.fit(df)
            # df = t.transform(df)
            df = t.fit(df).transform(df)
        return self
    
    def transform(self, df):
        df = df.copy()
        for t in self.transformers:
            df = t.transform(df)

        X = df.drop(columns=['median_house_value'])
        y = df['median_house_value'].copy() if 'median_house_value' in df.columns else None
        
        return X, y

In [ ]:
preprocessor = Preprocessor([FeatureCombinator(), MedianFiller()])
preprocessor.fit(train)
X_train, y_train = preprocessor.transform(train)
X_test, y_test = preprocessor.transform(test)

print(f'df["total_bedrooms"] balio galduak: {df['total_bedrooms'].isnull().sum()}')
print(f'X_train["bedrooms_ph"] balio galduak: {X_train['bedrooms_ph'].isnull().sum()}')
print(f'X_test["bedrooms_ph"] balio galduak: {X_test['bedrooms_ph'].isnull().sum()}')
#X_train
#y_train
#X_test
#y_test

Aurreprosezatzea beste momenturen batean erabili nahiko bagenu, gorde egin beharko genuke nolabait. `joblib` liburutegia erabili dezakegu:

In [ ]:
import joblib

# Aurreprosezatzailea fitxategi batean gorde
joblib.dump(preprocessor, "preprocessor.data")

Aurrerago, nahi dugunean berreskuratu dezakegu:

In [ ]:
# Aurreprosezatzailea fitxategi batetik kargatu
preprocessor2 = joblib.load("preprocessor.data")

X_test2, y_test2 = preprocessor2.transform(test)
#X_test2
#y_test2

## 8 - Ezaugarri kategorikoak bektorizatu

Eredu askok ezaugarri kuantitatiboak ametitzen dituzte soilik. Datubaseko **ocean_proximity** ezaugarria kategorikoa da: 

In [ ]:
df['ocean_proximity']

Eta denera 5 kategoria ezberdin ageri dira soilik:

In [ ]:
#set(train['ocean_proximity'])
df['ocean_proximity'].unique()

**One-Hot-Encoding**: $\{c_1, c_2, \dots , c_k\}$ kategoria edo balio posible dituen ezaugarri kategoriko bat aldagai kuantitatibo bilakatzeko modu sinple bat $k$ tamainako bektore bat sortzea da. Bektoreko posizio gehienen balioa $0$ izango da, bat izan ezik, aldagaiaren balioari dagokiona. Adibidez, aldagaiaren balioa $c_i$ denean, ordezkatuko dugun $v=[v_1,v_2,\dots,v_k]$ bektorearen balioa ondokoa da:
$$
v_j = \begin{cases} 
0, & j \ne i \\ 
1, & j = i 
\end{cases}
$$

Pandasek ba du ezaugarri kategorikoak erraz bektorizatzeko balio duen `pd.get_dummies()` funtzioa:

In [ ]:
pd.get_dummies(df['ocean_proximity'], dtype=int)

In [ ]:
pd.get_dummies(train, columns=['ocean_proximity'], dtype=int)

Hala ere, eraldaketa hau hasera-haseratik egin beharko litzateke, izan ere test-eko azpimultzoan kategoriaren bat faltako balitz, gauza arraroak gerta litezke. Adibidez, gure datubasean ez daude `'ISLAND'` balioa duten adibide asko: 

In [ ]:
print((df['ocean_proximity']=='ISLAND').sum())

eta kasualitatez denak entrenamendu azpimultzoan erori dira:

In [ ]:
print(f'Entrenamenduan: {(train['ocean_proximity']=='ISLAND').sum()}')
print(f'Testean: {(test['ocean_proximity']=='ISLAND').sum()}')

Hau dela eta, `pd.get_dummies()` funtzioa test-eko azpimultzoan aplikatzean, emaitzako dataframearen egitura ezberdina da (`15 columns` vs `16 columns`):

In [ ]:
pd.get_dummies(test, columns=['ocean_proximity'], dtype=int)

In [ ]:
print(pd.get_dummies(train, columns=['ocean_proximity'], dtype=int).shape)
print(pd.get_dummies(test, columns=['ocean_proximity'], dtype=int).shape)

Egoera hau ekiditeko, *Scikik-learn*-en `OneHotEncoder` kodetzailea erabili dezakegu. Kodetzaile honek entrenamenduko datuetan doitu daiteke (existitzen diren klaseak aztertu) eta ondoren beste edozein daturekin erabili:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# 1. Encoderra/Kodetzailea hasieratu
# sparse_output=False matrize dentsoa itzultzen du (beharrezkoa da Pandaserako)
# handle_unknown='ignore' testak kategoria berriak baditu, ez egin akatsik
# drop='first' lehenengo zutabea ezabatu erlazio linealik ez egoteko (dummy variable trap)
# feature_name_combiner=... sortutako zutabeen izenak ezartzen dituen funtzioa
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore',
    drop='first',
    feature_name_combiner=lambda column, cathegory : str(cathegory)
)

# Kodetzailearen emaitza Pandas-eko DataFrame izaten jarrai dezan
encoder.set_output(transform="pandas")

# 2. Encoderra doitu. OneHotEncoder-ak nx1 gisako matrize egitura espero du, df[['name']]
encoder.fit(X_train[['ocean_proximity']])

# 3. Kodetzailea aplikatu. OneHotEncoder-ak nx1 matrize egitura espero du, df[['name']]
encoder.transform(X_train[['ocean_proximity']]).head()

* `df['col_name']` &rarr; `pandas.Series ($n$ tamainako bektore egitura)
* `df[['col_name']]` (fancy indexing) &rarr; `pandas.DataFrame` ($n \times 1$ tamainako matrize egitura)

In [ ]:
print(f'{type(df['ocean_proximity']) = }')
print(f'{type(df[['ocean_proximity']]) = }')
print(f'{df['ocean_proximity'].shape = }')
print(f'{df[['ocean_proximity']].shape = }')

Sortutako zutabe berriak DataFrame zaharrarekin bildu behar dira eta *ocean_proximity* zutabea ezabatu:

In [ ]:
df1 = X_train.drop(columns=['ocean_proximity'])
df2 = encoder.transform(X_train[['ocean_proximity']])
pd.concat([df1,df2], axis=1).head()

OneHotEncoderra aurreprozesatzearen fase bat gehiago izango da, **entrenamenduko datuekin doitu neharko dena**, `total_bedrooms_median` parametroarekin gertatzen zen moduan:

In [ ]:
class CathegoryVectorizer:
    def __init__(self):
        self.ocean_proximity_encoder = OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
            drop='first',
            feature_name_combiner=lambda column, cathegory : str(cathegory)
        )
        self.ocean_proximity_encoder.set_output(transform="pandas")
        
    def fit(self, df):
        self.ocean_proximity_encoder.fit(df[['ocean_proximity']])
        return self
        
    def transform(self, df):
        df = pd.concat([
            df.drop(columns=['ocean_proximity']),
            self.ocean_proximity_encoder.transform(df[['ocean_proximity']])
        ], axis=1)
        return df

preprocessor = Preprocessor([FeatureCombinator(), MedianFiller(), CathegoryVectorizer()])
preprocessor.fit(train)
X_train, y_train = preprocessor.transform(train)
X_test, y_test = preprocessor.transform(test)

X_train
#y_train
#X_test
#y_test

### Hausnarketa
Zentzuzkoa al da `'ocean_proximity'=='ISLAND'` duten 5 lagin horiek mantentzea? ML eredua gai izango ote da 5 lagin horiek duten egitura barneratzeko? Gure kasuan ez dio gehiegi ajola, guztiak entrenamenduan erori direlako, baina laginak baztertzea ere aukera bat da. 

In [ ]:
print(f'Baztertuko diren laginak: {(df['ocean_proximity']=='ISLAND').sum()}')
print(f'{df.shape = }')
df = df[df['ocean_proximity']!='ISLAND']
print(f'ISLAND laginak DataFrame berrian: {(df['ocean_proximity']=='ISLAND').sum()}')
print(f'{df.shape = }')

Eta haseratik pausu guztiak berriro eman:

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

preprocessor = Preprocessor([FeatureCombinator(), MedianFiller(), CathegoryVectorizer()])
preprocessor.fit(train)
X_train, y_train = preprocessor.transform(train)
X_test, y_test = preprocessor.transform(test)

#X_train
#y_train
#X_test
#y_test

## 9 - Balio arraroak topatu

Noizbehinka interesgarria da balio arrarorik ote dugun konprobatzea. 

In [ ]:
X_train.drop(columns=['INLAND','NEAR BAY','NEAR OCEAN']).describe()

Lau kasu aztertuko ditugu:
* $max(bedrooms\_ph)=34.066667$
* $max(rooms\_ph)=132.533333$
* $min(bedrooms\_ph)=0.333333$
* $min(rooms\_ph)=0.888889$

Maximo/minimoak topatzeko `argmax()` eta `argmin()` funtzioak erabil ditzakegu

### 1. `max(bedrooms_ph)=34.066667`

In [ ]:
i = X_train['bedrooms_ph'].argmax()
print(f'indizea: {i}')
train.iloc[i]

Apur bat arraroa dirudi 15 etxeetan 511 logela egotea...

### 2. `max(rooms_ph)=132.533333`

In [ ]:
i = X_train['rooms_ph'].argmax()
print(f'indizea: {i}')
train.iloc[i]

Apur bat arraroa ere, 15 etxeetan 1988 gela egotea... Ohar bat: aurreko bi laginak berdinak dira, (`indizea: 5231`).

### 3. `min(bedrooms_ph)=0.333333`

In [ ]:
i = X_train['bedrooms_ph'].argmin()
print(f'indizea: {i}')
train.iloc[i]

9 etxebizitza, baina denera 3 logela. Logelarik gabeko etxebizitzak?

### 4. `min(rooms_ph)=0.888889`

In [ ]:
i = X_train['rooms_ph'].argmin()
print(f'indizea: {i}')
train.iloc[i]

36 etxebizitza, baina denera 32 gela. Gelarik gabeko etxebizitzak?

### Hausnarketa
Halako egoeretan datuak zuzenak ote diren galdetu beharko genioke gure buruari. Erroreak dituzten lagin guztiekin zerbait egin beharko genuke:
* Balioak ordezkatu
* Laginak baztertu

Kasu honetan, datuetan errorerik ez dagoela suposatuko dugu.

## 10 - Ezaugarri kuantitatiboak estandarizatu

Eredu batzuk ezaugarri kuantitatiboen eskalarekiko sentikortasun handia izan dezakete. 

In [ ]:
X_train.drop(columns=['INLAND','NEAR BAY','NEAR OCEAN']).hist(bins=50, figsize=(15,10));

**Estandarizazioa** (*Z-score*): Datuak eraldatzen ditu, 0 batez bestekoa eta 1 desbiderapen estandarra izan dezaten. $\mu$ datuen batez bestekoa bada eta $\sigma$ haien desbiderapen estandarra, orduan estandarizazioak ondoko transformazioa aplikatzen du:

$$x' = \frac{x - \mu}{\sigma}$$

Noiz erabili ohi da:
* Datuen banaketa normal baten araberakoak direnean.
* Ereduak banaketa normal bat espero duenean (**KONTUZ**, datuak gaussiarrak ez badira, emaitza ez da gausiarra izango)

*Scikit-learn*-eko StandardScaler erabili dezakegu estandarizazio transformazio bat zenbatetsi (entrenamenduan) eta edozein datu berritan aplikatzeko:

In [ ]:
from sklearn.preprocessing import StandardScaler

df_tmp = X_train[['median_income']]
scaler = StandardScaler() 
scaler.fit(df_tmp)
df_tmp['median_income_standarized'] = scaler.transform(df_tmp)
df_tmp.hist(bins=50, figsize=(10,3.3));

In [ ]:
X_train.iloc[0]

In [ ]:
class Standarizer:
    def __init__(self):
        columns = ['longitude','latitude','housing_median_age','population','households','median_income','rooms_ph','bedrooms_ph']
        self.standarizers = {column:StandardScaler() for column in columns}
        
    def fit(self, df):
        for column,scaler in self.standarizers.items():
            scaler.fit(df[[column]])
        return self
        
    def transform(self, df):
        for column,scaler in self.standarizers.items():
            df[column] = scaler.transform(df[[column]])
        return df

preprocessor = Preprocessor([FeatureCombinator(), MedianFiller(), CathegoryVectorizer(), Standarizer()])
preprocessor.fit(train)
X_train, y_train = preprocessor.transform(train)
X_test, y_test = preprocessor.transform(test)

X_train.hist(bins=50, figsize=(15,10));
#X_train
#y_train
#X_test
#y_test

### Hausnarketa
Helburu aldagaia ez dugu estandarizatu:

In [ ]:
pd.DataFrame(y_train).hist(bins=50, figsize=(5,3.3))

Hau arazo bat izan liteke eredu batzuentzat. Beste ezaugarri kuantitatiboekin jokatu dugun moduan, estandarizazio bat aplikatzea zentzuzkoa da, dena den horrek apur bat zailtzen du emaitzen interpretazioa eta beraz ez dugu egingo (estandarizazioa transformazio itzulgarria da, ez legoke arazo larririk).

## 11 - Ereduak sortu, doitu eta ebaluatu

Atal honetan `ml_course` moduluak eskaintzen dizkigun bi erregresio eredu erabiliko ditugu:
* `SimpleRegressor`: *A really simple regression model without hyper-parameters*
* `ComplexRegressor`: *A somehow complex regression model with some hyper-parameters*

Modulua instalatu eta bi ereduak inportatu:

In [ ]:
!pip install git+https://github.com/mpenagar/ml_course.git
import ml_course
from ml_course import SimpleRegressor, ComplexRegressor

In [ ]:
#?SimpleRegressor

In [ ]:
#?ComplexRegressor

Aurrerago erabiliko ditugun *Scikt-learn*-eko ereduek (`ml_course` moduluko ereduek bezalaxe) erabilera patroi sinple bat dute:

```python
model = Model(param1=, param2=, ...)  # 1 - Eredua sortu/hasieratu konfigurazio-parametroak ezarriz
model.fit(X, y)                       # 2 - Eredua doitu (barne-parametroak zenbatetsi)
y_predict = model.predict(X)          # 3 - Helburu-aldagaia iragarri
```

Gure kasuan, eta oraingoz ereduen konfigurazio-parametroak (**hiperparametroak**, *hyperparameters*) alde batera utziko ditugunez, baina entrenamenduko eta testeko datuak bananduta ditugunez, ondokoa egin dezakegu edozein eredurekin:

```python
model = Model(param1=, param2=, ...)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
```

`SimpleRegressor` ereduarentzat:

In [ ]:
model = SimpleRegressor()
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)

Ereduaren errendimendua aztertu aurretik, lortutako iragarpenen eta benetako balioen banaketak aztertu ditzakegu:

In [ ]:
y_train_df = pd.DataFrame({
    'real':y_train,
    'predict':y_train_predict,
})
sns.histplot(y_train_df)
plt.title('Target variable - Train');

Hainbat ohar:
* Eredua balio negatiboak iragartzeko gai da...
* Eredua entrenamenduko \$500k-etako atalasea gainditzeko gai da

Test-eko azpimultzoan portaera berdina atzematen da:

In [ ]:
y_test_df = pd.DataFrame({
    'real':y_test,
    'predict':y_test_predict
})
sns.histplot(y_test_df)
plt.title('Target variable - Test');

Behin emaitzak ditugula, iragarpenean izandako **errorea** zein izan den neurtu behar dugu. Errore honek gure ereduaren **Errendimendua** adieraziko du.  

### Errore koadratikoaren batez bestekoaren erroa (RMSE, *Root Mean Squared Error*)

Erregresio-atazetan eredu baten akatsa edo desbideratzea neurtzeko metrikarik erabiliena da. Ereduaren iragarpenen ($\hat{y}_i$) eta benetako balioen ($y_i$) arteko diferentzien distantzia (errorea) neurtzen du:

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}$$

Interpretazio erraz bat:

* **Akats handiekiko sentsibilitatea**: Desbideratzeak berretzen dituenez, RMSEak zigor handia ezartzen die akats handiei. Zenbat eta baxuagoa izan balioa, orduan eta zehatzagoa da eredua.
* Erro karratua egitean, lortutako emaitza **helburu-aldagaiaren unitate berdinetan** adierazten da ( etxebizitzen prezioa dolarretan badago, RMSEa ere dolarretan egongo da).

*Scikit-learn*-ek RMSE kalkulatzen duen `root_mean_squared_error` eskaintzen digu:

In [ ]:
from sklearn.metrics import root_mean_squared_error

print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Baina \$68k-etako errore bat... handia ala txikia al da? Konparaketak egin ahal izateko, interesgarria da *Baseline* erreferentzia-eredu bat izatea:
* **Ahalik eta sinpleenak** izan behar dira (*non-informative*), gure eredu *adimentsuarekin* konparatzeko.
* *Baseline* ereduarekiko **ezberdintasun txikia** &rarr; gure eredua ez da gai datuetan dagoen informazioa bereganatzeko.

Gure atazaren kasuan, adibidez, beti prezio berdina iragartzen duen eredu bat, *baseline* aproposa izan liteke.
* Zein balio, ordea?
* Entrenamenduan ikusitako *tarteto* balio bat &rarr; **mediana**.  

In [ ]:
y_median = y_train.median()
y_train_predict = np.full(y_train.shape,y_median)
y_test_predict = np.full(y_test.shape,y_median)

In [ ]:
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE:  ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Tira, badirudi ereduak zerbait ikasi duela!

Emaitzak taula batean jar ditzakegu:

| System          | $\mathbf{RMSE_{train}}$ | $\mathbf{RMSE_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline**        |      118546.68 |      119421.24 |
| **SimpleRegressor** |       68506.55 |       68567.85 |

Bigarren ereduarekin, `ComplexRegressor`, gauza bera egin dezakegu: 

In [ ]:
model = ComplexRegressor()
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)

In [ ]:
y_train_df = pd.DataFrame({
    'real':y_train,
    'predict':y_train_predict,
})
sns.histplot(y_train_df)
plt.title('Target variable - Train');

Zer gertatzen da hemen?
* Zergatik ez da iragarpenik ikusten?
* Nola da posible balio errealen histogramak iragarritako balioen histograma izkutatzea?
* Guztiz berdinak al dira?

Ea zer gertatu den testeko datuekin:

In [ ]:
y_test_df = pd.DataFrame({
    'real':y_test,
    'predict':y_test_predict
})
sns.histplot(y_test_df)
plt.title('Target variable - Test');

Testeko datuetan zerbait ikusten da... Ereduak nahiko ongi asmatzen duela dirudi, ezta?

In [ ]:
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Bitxia:
* Entrenamenduko errorea: **\$0**
* Testeko errorea: **\$69526.17** , `SimpleRegressor` ereduarena baina handiagoa.

&rarr; **Gaindoikuntza** (*Overfitting*)

Laburbilduz, badirudi eredu onena `SimpleRegressor` dela:

| System          | $\mathbf{RMSE_{train}}$ | $\mathbf{RMSE_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline**         |      118546.68 |      119421.24 |
| **SimpleRegressor**  |       68506.55 |       **68567.85** |
| **ComplexRegressor** |           0.00 |       69526.17 |

### Hausnarketa
**Errendimendu-neurri** gisa RMSE (*Root Mean Squared Error*) erabili dugu. Baina eta zein izan da ereduek erabilitako $\mathcal{L}$ **Galera-funtzioa**?

Erregresio ereduetan, ohikoa da galera funtzioa **Errore Koadratikoaren Batez bestekoa** (MSE, *Mean Squared Error*) izatea. Ereduaren dokumentazioak kontrakoa esaten ez badu, beti suposa dezakegu ereduaren barne parametroak MSE minimizatuz doitzen direla:

$$\boldsymbol{\theta}^* = \arg\min_{\boldsymbol{\theta}} \sum_{i=1}^{N} \mathcal{L}(y_i, \hat{y}_i) = \arg\min_{\boldsymbol{\theta}} MSE(y_i, \hat{y}_i)$$

Eta $RMSE = \sqrt{MSE}$ denez,

$$\boldsymbol{\theta}^* = \arg\min_{\boldsymbol{\theta}} RMSE(y_i, \hat{y}_i)^2 = \arg\min_{\boldsymbol{\theta}} RMSE(y_i, \hat{y}_i)$$

Eta beraz, <u>Errendimendu-neurria eta Galera-funtzioa bat datoz</u>.

## 12 - Ereduen hiperparametroak doitzen

Ereduek argumentuen bidez ezartzen diren konfigurazio-parametroak izan ditzakete. Parametro hauek ereduaren doitze prozesua kontrolatzen dute eta haren errendimenduan aldaketak sor ditzakete. Konfigurazio-parametro hauek **hiperparametro** izenaz ezagunak dira: 

```python
model = Model(param1=, param2=, ...)  # eredua hasieratu aukeratutako hiperparametroekin
```

Gure bi ereduetatik, soilik `ComplexRegressor` ereduak eskaintzen dizkigu hiperparametroak:

In [ ]:
#help(ComplexRegressor)

```
  Parameters
  ----------
  complexity : int, default=None
      Controls the maximum complexity of the model's internal structure. A higher value
      allows the model to learn more intricate patterns from the training data, but it
      heavily increases the risk of overfitting (memorizing the specific training
      examples instead of learning the general trend). If None, there is no limit on
      how complex the model can become (use with caution!).

  min_split_size : int, default=2
      The minimum amount of data points required before the model is allowed to divide
      a group into even smaller pieces. Increasing this value stops the model from
      creating overly complex and specific rules for very small sets of data.

  random_state : int, default=None
      Controls the randomness of the estimator. To obtain a deterministic behaviour
      during fitting, random_state has to be fixed to an integer
```

Badirudi `complexity` eta `min_split_size` hiperparametroen defektuzko balioak aurrez ikusitako gaindoikuntzaren  (*Overfitting*) iturria izan daitezkeela. Egin dezagun froga sinple bat: 

In [ ]:
model = ComplexRegressor(complexity=10, min_split_size=10, random_state=42)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

* Entrenamenduko errorea: 0-tik 48001.08-ra igo da
* Testeko errorea: 69526.17-tik 60833.57-ra jeitsi da, orain arteko emaitzarik hoberena!

Hala ere, bi arazo berri sortzen zaizkigu:
1. Bi hiperparametroren konbinaketa optimoa topatu behar dugu
2. Ezin ditugu testeko datuak erabili konbinaketa hoberena zein izango ote den erabakitzeko, eta entrenamendua erabiltzen badugu, ez dugu aukeraketa egokia egingo (goidoikuntza).

## 13 - Baliozkotze azpimultzoa

Testeko datuak erabili ezin ditzakegunez, aukera argi bat entrenamenduko datuak berriro bi azpimultzotan banatzea litzateke:
* **X_train2, y_train2**: eredua entrenatzeko erabiliko duguna
* **X_valid, y_valid**: baliozkotze azpimultzoa, test azpimultzoa balitz bezala, hiperparametro konbinaketa hoberena aukeratzeko erabiliko duguna.

In [ ]:
X_train2, X_valid = train_test_split(X_train, test_size=0.2, random_state=42)
y_train2, y_valid = train_test_split(y_train, test_size=0.2, random_state=42)
print(f'{X_train.shape = }')
print(f'{X_train2.shape = }')
print(f'{X_valid.shape = }')
print(f'{X_test.shape = }')

**OHARRA:** `X_train,y_train` aldagaietatik abiatzen bagara, **beharrezkoa da** `random_state` argumentuaren balioa berdina izatea, datuen koherentzia bermatzeko. Hau bera `train` aldagaiatik abiatuz egin izan bagenu, ez legoke arrisku hau (`X,y` banaketa aurreprozesatzeak egingo luke):

```python
train2, valid = train_test_split(train, test_size=0.2, random_state=42)
X_train2, y_train2 = preprocessor.transform(train2)
X_valid, y_valid = preprocessor.transform(valid)
```

## 14 - Hiperparametroen optimizazioa (*Grid Search*)

Baliozkotze azpimultzoa definituta dugularik, hiperparametroen edozein konbinaketak izango duen errendimendua bertan neurtu dezakegu. Hala er, orohar, **hiperparametroen bilaketa espazioa mugagabea izango da**. Hau dela eta, konbinaketa kopuru finitu batekin konformatu beharko gara.

Hiperparametro bakoitzarentzat balio multzo finitu bat definitzen badugu, konbinaketa guztiek sareta (*grid*) bat osotuko dute. Sareta horretako konbinaketa bakoitzarekin eredua doitu (`train2` erabiliz) eta bere errendimendua neurtu dezakegu (`valid` erabiliz). Hiperparametroak optimizatzeko prozesu honi **Sareta bidezko Bilaketa** (*Grid Search*) deritzo.

Bilaketa sareta hiztegi baten bidez adierazi dezakegu, gakoak parametro izenak izanik eta balioak parametroak har ditzakeen balio multzo finitua adierazten duen zerrenda:

```python
grid = {
    'param1` : [value1, value2, ...] ,
    'param2` : [value1, value2, ...] ,
    ...
    'paramk` : [value1, value2, ...]
}    

Sortu dezagun lehenenik aurreko `grid` moduko hiztegi bat jaso eta konbinaketa guztiak dituen hiztegi zerrenda bueltatuko duen funtzioa:

In [ ]:
def expand_grid(grid):
    if not grid:
        return [{}]

    grid = grid.copy()
    name, values = grid.popitem()   
    other_expanded = expand_grid(grid)
    
    result = []
    for v in values:
        for combo in other_expanded:
            result.append({**combo, name: v})
            
    return result

Froga dezagun adibide sinple batekin

In [ ]:
x = {'a': [1, 2], 'b': [3], 'c': [4, 5, 6]}

for d in expand_grid(x):
    print(d)    

`expand_grid()` funtzioa dugularik, egin behar dugun guztia konbinaketak banan banan frogatzea besterik ez da:

In [ ]:
def grid_search(model_class, grid, X_train, y_train, X_valid, y_valid):
    best_params = None
    best_rmse = np.inf
    for params in expand_grid(grid):
        model = model_class(**params)
        model.fit(X_train, y_train)
        y_valid_predict = model.predict(X_valid)
        rmse = root_mean_squared_error(y_valid, y_valid_predict)
        print(f'{params=}  RMSE: {rmse:.2f}')
        if rmse < best_rmse :
            best_rmse = rmse
            best_params = params
    return best_params, best_rmse

In [ ]:
grid = {
    'complexity' : [5,10,15,20] ,
    'min_split_size' : [2, 4, 8, 16, 32, 64] ,
    'random_state' : [42]
}

best_params, best_rmse = grid_search(ComplexRegressor, grid, X_train2, y_train2, X_valid, y_valid)
print(f'{best_params = }')
print(f'{best_rmse = }')

Test azpimultzoa ikutzen ari ez garenez, nahi dugun adina saiakera egin ditzakegu, bilaketa-saretako tarteak berdefinituz:

In [ ]:
grid = {
    'complexity' : [13,14,15,16,17] ,
    'min_split_size' : [40,50,64,80,90] ,
    'random_state' : [42]
}

best_params, best_rmse = grid_search(ComplexRegressor, grid, X_train2, y_train2, X_valid, y_valid)
print(f'{best_params = }')
print(f'{best_rmse = }')

In [ ]:
grid = {
    'complexity' : [15,16,17] ,
    'min_split_size' : [45,47,50,52,55] ,
    'random_state' : [42]
}

best_params, best_rmse = grid_search(ComplexRegressor, grid, X_train2, y_train2, X_valid, y_valid)
print(f'{best_params = }')
print(f'{best_rmse = }')

In [ ]:
grid = {
    'complexity' : [15,16,17] ,
    'min_split_size' : [48,49,50,51,52] ,
    'random_state' : [42]
}

best_params, best_rmse = grid_search(ComplexRegressor, grid, X_train2, y_train2, X_valid, y_valid)
print(f'{best_params = }')
print(f'{best_rmse = }')

Hiperparametroen doikuntza ontzat ematen badugu, orain testean duen errendimendua neurtzea besterik ez da falta. Eredua berriro doituko dugu, orain <u>entrenamenduko datu guztiekin</u> eta *Grid Search*-ak topatutako <u>hiperparametro hoberenekin</u>:

In [ ]:
model = ComplexRegressor(**best_params)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Hiperparametroak doitu ostean, `ComplexRegressor` izan da garailea:

| System          | $\mathbf{RMSE_{train}}$ | $\mathbf{RMSE_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline**         |      118546.68 |      119421.24 |
| **SimpleRegressor**  |       68506.55 |       68567.85 |
| **ComplexRegressor** |       44934.96 |       **59756.76** |

## 15 - Grid Search Cross Validation

`grid_search` inplementatzea erronka interesgarria izan bada ere, *Scikit-learn*-ek badu hiperparametroen sareta bilaketa egiteko funtzio aproposa, `GridSearchCV`. Funtzio honek beste ezaugarri interesgarri bat gehitzen du gainera: **k-fold Cross Validation** deritzona.

**k-fold Baliozkotze Gurutzatua** (*k-fold Cross Validation*): Eredu baten errendimendua modu fidagarrian ebaluatzeko teknika bat da, eskura dugun datu-multzoari ahalik eta etekin handiena ateraz eta zoriak ebaluazioan izan dezakeen eragina murriztuz. Ezaugarri nagusiak:
* **Banaketa**: Datu-multzoa tamaina bereko $k$ azpimultzotan banatzen da.
* **Txandakatzea**: Prozesua $k$ aldiz errepikatzen da. Iterazio bakoitzean, azpimultzoetako bat baliozkotzeko erabiltzen da eta gainerako $k-1$ azpimultzoak eredua entrenatzeko.
* **Batez bestekoa**: Iterazio guztietako emaitzak jaso eta guztien batez bestekoa kalkulatzen da. Batez besteko hori izango da ereduaren azken metrika ofiziala.

Ondoko irudiak *5-fold Baliozkotze Gurutzatu* baten prozedura laburtzen du:

<img src="img/k-fold-cross-validation.png" style="display: block; margin: 0 auto; width: 80%; height: auto;" alt="k-fold cross validation"/>

*Scikit-learn*-en `GridSearchCV` funtzioa guk sortutako `grid_search` funtzioaren antzekoa bada era, badira ezberdintasun batzuk:

`grid_search = GridSearchCV(estimator=... , param_grid=... , cv=... , scoring=...)`

* `estimator`: Erabiliko duen eredua. Ez da eredu clasea baizin eta eredu instantzia bat. Horrela, *Grid Search*-ean aldatuko ez diren hiperparametro finkoak ezarri ditzakegu.
* `param_grid`: Gure funtzioaren berdina, hiperparametroen bilaketa sareta, hiztegi batez adierazia
* `cv`: Baliozkotze gurutzatuaren $k$ aldagaiaren balioa
* `scoring`: ereduaren errendimendua konparatzeko erabiliko den errendimendu neurria. Geroz eta score handiagoa, are eredu hobea.

Auurez egindako frogetatik abiatuz, ondokoa froga genezake:

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {
    'complexity' : [14,15,16,17,18] ,
    'min_split_size' : [47,48,49,50,51] ,
}

grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error' #negative, errendimendua baita, ez galera.
)

grid_search.fit(X_train, y_train)

`grid_search` objektuak eremu asko ditu, besteak beste:

In [ ]:
grid_search.best_params_ , (-grid_search.best_score_)**0.5

Doitutako hiperparametroekin eredua berriro doitu eta testean errendimendua neurtu dezakegu:

In [ ]:
model = ComplexRegressor(random_state=42, **grid_search.best_params_)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Hala ere, ez da beharrezkoa eredua berriro entrenatzea, izan ere `grid_search.best_estimator_` eremuak hiperparametro hoberenekin entrenatutako eredua du dagoeneko:

In [ ]:
model = grid_search.best_estimator_
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

Proiektuaren emaitza finalak ondokoak lirateke:

| System          | $\mathbf{RMSE_{train}}$ | $\mathbf{RMSE_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline**         |      118546.68 |      119421.24 |
| **SimpleRegressor**  |       68506.55 |       68567.85 |
| **ComplexRegressor** |       44466.62 |       **59653.15** |

### Hausnarketa

Kasu honetan hobekuntza txiki bat egon bada ere, *Grid Search*ean baliozkotze gurutzatua erabiltzeak ez du zertan amaierako testaren errendimendua hobetu beharrik. Posible litzateke jasotako azken emaitza aurrekoa baino apur bat okerragoa izatea. Hala eta guztiz ere, baliozkotze gurutzatuak segurtasun handiagoa eskaintzen du hiperparametroen doikuntzan eta zorizko egoerak baztertuz.